## Imports Library

In [24]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras import layers


## Load Processed CSV Files

In [25]:
DATA_PATH="/kaggle/input/datasets/sourabhsaxena/new-data"

In [26]:
train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_PATH, "val.csv"))
test_df = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

with open(os.path.join(DATA_PATH,"class_names.json"),"r") as f:
    class_names=json.load(f)

num_classes=len(class_names)


print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("Classes:", class_names)

Train: (7010, 9)
Validation: (1502, 9)
Test: (1503, 9)
Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


## Define Image Parameters

In [28]:
IMG_SIZE=224
BATCH_SIZE=32
AUTOTUNE=tf.data.AUTOTUNE

## Image Loading Function

In [29]:
def load_image(image_path,label):
    image=tf.io.read_file(image_path)
    image=tf.image.decode_jpeg(image,channels=3)
    image=tf.image.resize(image,[IMG_SIZE,IMG_SIZE])
    image=image/255.0 #normalize to[0,1]
    return image,label
    

## Create tf.data Dataset

In [30]:
def create_dataset(df,training=False):
    image_paths=df['image_path'].values
    labels=df['label'].values

    datset=tf.data.Dataset.from_tensor_slices((image_paths,labels))

    dataset=dataset.map(load_image,num_parallel_calls=AUTOTUNE)

    if training:
        dataset=dataset.shuffle(buffer_size=1000)
    dataset=dataset,batch(BATCH_SIZE)
    dataset=datset.prefetch(AUTOTUNE)
    return dataset

## Data Augmentation

In [31]:
data_augmentation=keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
])

In [32]:
def create_dataset(df, training=False):
    image_paths = df['image_path'].values
    labels = df['label'].values
    
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    
    dataset = dataset.map(load_image, num_parallel_calls=AUTOTUNE)
    
    if training:
        dataset = dataset.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=AUTOTUNE
        )
        dataset = dataset.shuffle(1000)
    
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)
    
    return dataset


## Create Final Datasets

In [33]:
train_dataset=create_dataset(train_df,training=True)
val_dataset=create_dataset(val_df,training=False)
test_dataset=create_dataset(test_df,training=False)

print("Datsets ready")

Datsets ready
